# Project: PCA and a softmax classifier with NumPy only

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The goal
In this project you put everything together and build two real algorithms **without scikit-learn**. The data: iris flowers (150 samples, 4 features, 3 classes).

| Part | Math used | Lesson |
|---|---|---|
| Standardization | Mean, standard deviation, z-score | Descriptive statistics |
| PCA | Covariance matrix, eigenvectors, SVD | Eigenvalues, SVD |
| The classifier | Matrix products, softmax | Matrices, math of classification |
| The loss | Cross-entropy | Likelihood, entropy |
| Training | Gradient, gradient descent | Partial derivatives, gradient descent |
| Checking | Numerical derivative | Derivatives |

## Part 1: PCA in four steps
| Step | Formula |
|---|---|
| 1. Standardize | $Z=(X-\mu)/\sigma$ |
| 2. Covariance matrix | $C=\dfrac{1}{n-1}Z^\top Z$ |
| 3. Eigendecomposition | $C=V\Lambda V^\top$, sorted in decreasing order |
| 4. Project | $T=ZV_k$ |

The share of variance of each component is $\lambda_i/\sum_j\lambda_j$.

> [Left: the four iris features projected onto two principal components. Right: the share of variance of each component.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/math_pca_iris.png)

*Left: the four iris features projected onto two principal components. Right: the share of variance of each component.*



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris          # used only to load the data

iris = load_iris(); X, y = iris.data, iris.target
Z = (X - X.mean(axis=0)) / X.std(axis=0)                 # 1. standardize
C = Z.T @ Z / (len(Z) - 1)                               # 2. covariance matrix
vals, vecs = np.linalg.eigh(C)                           # 3. eigendecomposition (ascending)
order = np.argsort(vals)[::-1]; vals, vecs = vals[order], vecs[:, order]
T = Z @ vecs[:, :2]                                      # 4. project on 2 components

print("eigenvalues:", vals.round(3))
print("variance explained:", (vals / vals.sum()).round(3), " first two:", round(vals[:2].sum() / vals.sum(), 3))
print("first component:", vecs[:, 0].round(3))

# The SVD gives the same eigenvalues
s = np.linalg.svd(Z, compute_uv=False)
print("from the SVD:", (s ** 2 / (len(Z) - 1)).round(3))

for k, name in enumerate(iris.target_names):
    plt.scatter(T[y == k, 0], T[y == k, 1], s=15, label=name)
plt.xlabel("PC 1"); plt.ylabel("PC 2"); plt.legend(); plt.title("PCA from scratch"); plt.show()



The first two components keep about 96% of the variance. The SVD gives the same answer: $\lambda_i=\sigma_i^2/(n-1)$.

## Part 2: softmax regression
The model for n samples, d features and K classes:

$$Z=XW+b\qquad P=\mathrm{softmax}(Z)\qquad L=-\frac1n\sum_i\ln P_{i,y_i}$$

The gradients (from the classification-math lesson, in matrix form):

$$\frac{\partial L}{\partial W}=\frac1nX^\top(P-Y)\qquad\frac{\partial L}{\partial b}=\frac1n\sum_i(P_i-Y_i)$$

where Y is the one-hot matrix of the labels.

| Symbol | Shape | In this project |
|---|---|---|
| X | $n\times d$ | $120\times4$ |
| W | $d\times K$ | $4\times3$ |
| b | K | 3 |
| P and Y | $n\times K$ | $120\times3$ |



In [ ]:
import numpy as np
from sklearn.datasets import load_iris

iris = load_iris(); X, y = iris.data, iris.target
rng = np.random.default_rng(0)
idx = rng.permutation(150); tr, te = idx[:120], idx[120:]
mu, sd = X[tr].mean(axis=0), X[tr].std(axis=0)           # statistics from the training set only
Xtr, Xte = (X[tr] - mu) / sd, (X[te] - mu) / sd
Y = np.eye(3)[y[tr]]                                     # one-hot labels, 120 x 3

def softmax(Z):
    E = np.exp(Z - Z.max(axis=1, keepdims=True))
    return E / E.sum(axis=1, keepdims=True)

def loss_and_grads(W, b):
    P = softmax(Xtr @ W + b)
    loss = -np.mean(np.log(P[np.arange(120), y[tr]]))
    return loss, Xtr.T @ (P - Y) / 120, (P - Y).mean(axis=0)

W = np.zeros((4, 3)); b = np.zeros(3)
print("initial loss:", round(loss_and_grads(W, b)[0], 4), "  ln 3 =", round(np.log(3), 4))

# Gradient check on one weight, at a random point
Wr = rng.normal(0, 0.5, (4, 3)); eps = 1e-6
Wp = Wr.copy(); Wp[1, 2] += eps; Wm = Wr.copy(); Wm[1, 2] -= eps
numeric = (loss_and_grads(Wp, b)[0] - loss_and_grads(Wm, b)[0]) / (2 * eps)
print("gradient check:", round(numeric, 6), "vs", round(loss_and_grads(Wr, b)[1][1, 2], 6))

for epoch in range(1, 501):
    loss, gW, gb = loss_and_grads(W, b)
    W -= 0.5 * gW; b -= 0.5 * gb
    if epoch in (1, 10, 100, 500): print(f"epoch {epoch:3d}: loss = {loss:.4f}")

acc = lambda A, t: np.mean(np.argmax(softmax(A @ W + b), axis=1) == t)
print("train accuracy:", round(acc(Xtr, y[tr]), 3), "  test accuracy:", round(acc(Xte, y[te]), 3))

from sklearn.linear_model import LogisticRegression
print("scikit-learn test accuracy:", round(LogisticRegression(max_iter=1000).fit(Xtr, y[tr]).score(Xte, y[te]), 3))



## What to check
- The initial loss should be close to $\ln3=1.099$ (the model knows nothing yet and gives each class one third).
- The analytic and numerical gradients should agree to several decimals.
- The loss should fall every epoch; if it rises, the learning rate is too large.
- The final accuracy should be close to scikit-learn's.

## Keep going
- Add a hidden ReLU layer and write its backpropagation.
- Add an L2 penalty to the loss and the gradient: $\lambda\lVert W\rVert^2$ and $2\lambda W$.
- Replace plain gradient descent with Adam.
- Train the same classifier on the two PCA components and compare the accuracy.